In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1r-gcU5M41pn3LmL2W3H6ZNR3f27F-qEa", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Tool Distribution & tool_choice

*Part 3 of the Vizuara series on Tool Design & MCP Integration*
*Estimated time: 45 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/tool-design-mcp/practice/3/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you hand a new employee a Swiss Army knife with 50 blades and say "fix the computer." They will spend more time fumbling through blades than actually fixing anything. Now imagine you hand them a screwdriver, a pair of pliers, and a cable tester. Three tools, each clearly suited to a specific task. They get to work immediately.

This is exactly what happens with AI agents. Give an agent 18 tools and it struggles to select the right one. Give it 4-5 focused tools and selection becomes reliable.

In this notebook, we will explore tool distribution across agents, learn the empirical sweet spot for tool counts, and master the three `tool_choice` modes: auto, any, and forced.

In [ ]:
# Setup
import json
import random
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from collections import Counter

random.seed(42)
np.random.seed(42)

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Intuition Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_intuition_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Let us simulate what happens to tool selection accuracy as we increase the number of available tools.

In [ ]:
def simulate_selection_accuracy(n_tools, n_queries=100, quality=0.8):
    correct = 0
    for _ in range(n_queries):
        scores = np.random.uniform(0, 1, n_tools)
        scores[0] += quality  # correct tool gets boost
        if np.argmax(scores) == 0:
            correct += 1
    return correct / n_queries

tool_counts = [2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 25]
accuracies = [simulate_selection_accuracy(n) for n in tool_counts]

print("TOOL COUNT vs SELECTION ACCURACY")
print("=" * 45)
for n, acc in zip(tool_counts, accuracies):
    assessment = "Excellent" if acc >= 0.95 else "Good" if acc >= 0.85 else "Acceptable" if acc >= 0.75 else "DEGRADED"
    bar = "#" * int(acc * 30)
    print(f"{n:>6} | {acc:>7.1%} | {bar} {assessment}")

print("\nKey finding: 4-5 tools maintains >90% accuracy.")

In [ ]:
#@title 🎧 Listen: Degradation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_degradation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Selection Degradation Problem

In [ ]:
@dataclass
class AgentTool:
    name: str
    description: str
    domain: str

# A bloated agent with 12 tools
bloated = [
    AgentTool("get_customer", "Get customer info", "triage"),
    AgentTool("classify_intent", "Classify intent", "triage"),
    AgentTool("route_request", "Route request", "triage"),
    AgentTool("lookup_order", "Look up order", "orders"),
    AgentTool("track_shipment", "Track shipment", "orders"),
    AgentTool("process_return", "Process return", "orders"),
    AgentTool("cancel_order", "Cancel order", "orders"),
    AgentTool("get_invoice", "Get invoice", "billing"),
    AgentTool("process_refund", "Process refund", "billing"),
    AgentTool("apply_credit", "Apply credit", "billing"),
    AgentTool("update_payment", "Update payment", "billing"),
    AgentTool("escalate_to_human", "Escalate", "all"),
]

def simulate_cross_domain(toolset, query_domain, n_trials=200):
    wrong = sum(1 for _ in range(n_trials)
                if random.choice(toolset).domain not in (query_domain, "all"))
    return wrong / n_trials

print("CROSS-DOMAIN MISUSE with 12-tool agent")
print("=" * 45)
for domain in ["triage", "orders", "billing"]:
    rate = simulate_cross_domain(bloated, domain)
    print(f"  {domain:>10} queries: {rate:.0%} selected wrong domain")

print("\nWith all tools in one agent, cross-domain misuse is rampant.")

In [ ]:
#@title 🎧 Listen: Scoping Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_scoping_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Hands-On: Scoping Tools by Agent Role

In [ ]:
# Specialized agents with scoped tools
agents_config = {
    "Triage Agent": {
        "tools": ["get_customer", "classify_intent", "route_to_specialist"],
        "count": 3,
    },
    "Order Specialist": {
        "tools": ["lookup_order", "track_shipment", "process_return", "escalate_to_human"],
        "count": 4,
    },
    "Billing Specialist": {
        "tools": ["get_invoice", "process_refund", "apply_credit", "escalate_to_human"],
        "count": 4,
    },
}

print("SCOPED AGENT ARCHITECTURE")
print("=" * 55)
for name, config in agents_config.items():
    print(f"\n{name} ({config['count']} tools):")
    for t in config["tools"]:
        print(f"  - {t}")

total = sum(c["count"] for c in agents_config.values())
max_per = max(c["count"] for c in agents_config.values())
print(f"\nTotal tools: {total} | Max per agent: {max_per}")
print("The triage agent CANNOT call process_refund.")
print("The billing agent CANNOT call track_shipment.")
print("This is exactly what we want.")

In [ ]:
#@title 🎧 Listen: Constrained Tools
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_constrained_tools.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Replacing Generic with Constrained Tools

In [ ]:
replacements = [
    ("fetch_url", "load_document", "Only accepts URLs matching docs.company.com"),
    ("execute_code", "run_python_sandbox", "Isolated sandbox with 30s timeout"),
    ("send_message", "post_to_channel", "Only approved Slack channels"),
    ("database_query", "read_customer_table", "Read-only, customers table only"),
]

print("GENERIC vs CONSTRAINED TOOLS")
print("=" * 55)
for generic, constrained, constraint in replacements:
    print(f"  {generic:>20} ==> {constrained}")
    print(f"  {'':>20}     Constraint: {constraint}")
    print()

print("Constrained tools enforce boundaries at the tool level,")
print("so the model cannot wander outside its intended scope.")

In [ ]:
#@title 🎧 Listen: Toolchoice Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_toolchoice_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. tool_choice Configuration

In [ ]:
modes = {
    "auto": {
        "config": '{"type": "auto"}',
        "behavior": "Model decides IF and WHICH tool to call",
        "use_when": "Standard agent interactions",
    },
    "any": {
        "config": '{"type": "any"}',
        "behavior": "Model MUST call a tool (no text-only response)",
        "use_when": "Guarantee action, prevent chatting",
    },
    "forced": {
        "config": '{"type": "tool", "name": "extract_metadata"}',
        "behavior": "Model MUST call the SPECIFIED tool",
        "use_when": "Enforce step ordering in pipelines",
    },
}

print("tool_choice CONFIGURATION")
print("=" * 60)
for mode, info in modes.items():
    print(f"\nMode: {mode.upper()}")
    print(f"  Config:   {info['config']}")
    print(f"  Behavior: {info['behavior']}")
    print(f"  Use when: {info['use_when']}")

# Pipeline demo
print("\n" + "=" * 60)
print("\nFORCED ORDERING EXAMPLE:")
steps = [
    (1, "FORCED", "extract_metadata", "type=invoice, pages=3, lang=en"),
    (2, "AUTO", "classify_document", "category=accounts_payable"),
    (3, "AUTO", "summarize", "Invoice #1234, $5,000, due 2025-02-01"),
    (4, "AUTO", "store_result", "Stored as DOC-001"),
]
for turn, mode, tool, result in steps:
    print(f"  Turn {turn} [{mode:>6}] {tool}: {result}")

In [ ]:
#@title 🎧 Listen: Research Exercise
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_research_exercise.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. TODO Exercise

Design tool sets for a 4-agent research system. Each agent should have 3-5 tools.

In [ ]:
available_tools = [
    "search_web", "search_academic_papers", "fetch_webpage",
    "extract_key_findings", "compare_sources", "identify_contradictions",
    "write_summary", "generate_citations", "format_bibliography",
    "verify_fact", "check_source_reliability", "handoff_to_coordinator",
]

# TODO: Assign tools to each agent (3-5 per agent)
agents = {
    "Web Searcher": [],       # TODO
    "Document Analyzer": [],  # TODO
    "Report Writer": [],      # TODO
    "Fact Checker": [],       # TODO
}

print("RESEARCH SYSTEM TOOL ASSIGNMENT")
print("=" * 55)
all_assigned = []
for name, tools in agents.items():
    count = len(tools)
    status = "OK" if 3 <= count <= 5 else "NEEDS WORK"
    print(f"\n{name} ({count} tools) [{status}]:")
    for t in tools:
        print(f"  - {t}")
    all_assigned.extend(tools)

unassigned = set(available_tools) - set(all_assigned)
if unassigned:
    print(f"\nUnassigned: {unassigned}")
else:
    print("\nAll tools assigned!")

In [ ]:
#@title 🎧 Listen: Pipeline Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_pipeline_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Mini-Project: Document Processing Pipeline

In [ ]:
class DocumentPipeline:
    def __init__(self):
        self.processed = []
    
    def process(self, doc):
        steps = []
        # Forced: metadata first
        meta = {"title": doc["title"], "words": len(doc["content"].split()),
                "type": "report" if "quarterly" in doc["content"].lower() else "memo"}
        steps.append(("FORCED", "extract_metadata", meta))
        
        # Auto: classify
        cls = "financial" if meta["type"] == "report" else "general"
        steps.append(("AUTO", "classify", cls))
        
        # Auto: summarize
        summary = f"{doc['title']}: {doc['content'][:60]}..."
        steps.append(("AUTO", "summarize", summary))
        
        # Auto: store
        doc_id = f"DOC-{len(self.processed)+1:03d}"
        self.processed.append({"id": doc_id, "title": doc["title"], "class": cls})
        steps.append(("AUTO", "store", doc_id))
        
        return steps

pipeline = DocumentPipeline()
docs = [
    {"title": "Q4 Earnings", "content": "Quarterly revenue increased 15% year-over-year to $2.3B in the fourth quarter"},
    {"title": "Team Update", "content": "Quick note: standup moved to 10am starting Monday"},
    {"title": "Migration Plan", "content": "Phase 1 involves migrating the database to PostgreSQL over six weeks"},
]

print("BATCH DOCUMENT PROCESSING")
print("=" * 60)
for doc in docs:
    print(f"\n--- {doc['title']} ---")
    for mode, tool, result in pipeline.process(doc):
        out = json.dumps(result) if isinstance(result, dict) else str(result)
        print(f"  [{mode:>6}] {tool}: {out[:70]}")

print(f"\nProcessed: {len(pipeline.processed)} documents")
for r in pipeline.processed:
    print(f"  {r['id']}: {r['title']} [{r['class']}]")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Key Takeaways & What's Next

1. **4-5 tools per agent** is the reliability sweet spot
2. **Scope tools by role** to eliminate cross-domain misuse
3. **Replace generic with constrained tools** for safety
4. **tool_choice: auto** for standard, **any** for forced action, **forced** for ordering
5. **Forced selection** ensures critical steps happen first in pipelines

In [ ]:
print("Notebook complete!")
print()
print("Key skills practiced:")
print("  1. Measuring selection accuracy vs tool count")
print("  2. Scoping tools by agent role")
print("  3. Replacing generic with constrained tools")
print("  4. Using auto, any, and forced tool_choice modes")
print("  5. Building pipelines with forced ordering")
print()
print("Next: Notebook 4 - MCP Server Configuration & Built-in Tools")